# Binary Classification with a Small CNN and Grad-CAM

A from-scratch convolutional net on PneumoniaMNIST (28x28 grayscale), with the same
evaluation setup as the full-resolution chest study: val-AUC-based selection, early
stopping, a tuned decision threshold, and a Grad-CAM heatmap. Because the images are only
28x28 (center-cropped and normalized by MedMNIST), the localization is coarse, but the arc
mirrors chest [06](../chest-x-ray-images-pneumonia/06_mobilenetv2_transferring.ipynb).

## Setup

In [ ]:
# One-time setup: make the `visualization` helper importable, then fetch data +
# resolve paths. Each study's fetch logic lives in its own download_data.py.
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.isdir("ConvolutedComputerVision"):
        !git clone -q https://github.com/samlowe106/ConvolutedComputerVision.git
    sys.path.insert(0, "ConvolutedComputerVision/src")

from visualization import colab_bootstrap

DATA_ROOT, CKPT_ROOT = colab_bootstrap(study="pneumonia_mnist")

In [ ]:
import datetime

import numpy as np
import tensorflow as tf
from sklearn.metrics import precision_recall_curve

notebook_start_time = datetime.datetime.now()

In [ ]:
# load the MedMNIST PneumoniaMNIST .npz (fetched by download_data.py via the
# colab_bootstrap cell). Each split is (N, 28, 28) uint8 images + {0: normal, 1: pneumonia}.
_npz = np.load(os.path.join(DATA_ROOT, "pneumoniamnist.npz"))


def _make_ds(split):
    images = _npz[f"{split}_images"][..., None]  # add channel -> (N, 28, 28, 1)
    labels = _npz[f"{split}_labels"].reshape(-1).astype("int32")
    return tf.data.Dataset.from_tensor_slices((images, labels))


batch_size = 32
train_ds = _make_ds("train").batch(batch_size).prefetch(tf.data.AUTOTUNE)
validation_ds = _make_ds("val").batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_ds = _make_ds("test").batch(batch_size).prefetch(tf.data.AUTOTUNE)

y_val = np.concatenate([y for _, y in validation_ds], axis=0)
y_test = np.concatenate([y for _, y in test_ds], axis=0)

In [ ]:
from visualization import (
    reset_keras,
    show_confusion_matrix,
    show_gradcam,
    summary_graphics,
)


def get_class_training_weights(train_ds, normalize=True):
    labels, counts = np.unique(
        np.concatenate([y for x, y in train_ds], axis=0), return_counts=True
    )
    total = sum(counts)
    weights = [total / (2 * count) for count in counts]
    max_weight = np.max(weights)
    if normalize:
        return {label: weights[i] / max_weight for i, label in enumerate(labels)}
    return {label: weights[i] for i, label in enumerate(labels)}


class_weight = get_class_training_weights(train_ds)
print(f"Weight for normal class: {class_weight[0]:1.3f}")
print(f"Weight for pneumonia class: {class_weight[1]:1.3f}")

## Model

In [ ]:
metrics = [
    tf.keras.metrics.TruePositives(name="tp"),
    tf.keras.metrics.TrueNegatives(name="tn"),
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.AUC(name="auc"),
]


def make_callbacks(filepath):
    # select on val AUC (threshold-free, robust to the 74/26 class imbalance)
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=filepath, save_best_only=True, monitor="val_auc", mode="max"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=6, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min", factor=0.5, patience=3
        ),
    ]


epochs = 25

In [ ]:
def build_cnn():
    # a small conv backbone with a GlobalAveragePooling head. No horizontal flip, since
    # chest anatomy is left/right specific. Rescaling lives inside the model so raw uint8
    # images (and Grad-CAM on raw images) work directly.
    return tf.keras.models.Sequential(
        [
            tf.keras.layers.Input((28, 28, 1), name="input"),
            tf.keras.layers.RandomRotation(0.1),
            tf.keras.layers.RandomZoom(0.1),
            tf.keras.layers.Rescaling(1.0 / 255),
            tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.SeparableConv2D(64, 3, padding="same", activation="relu"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.SeparableConv2D(64, 3, padding="same", activation="relu"),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
        ],
        name="pneumonia_mnist_cnn",
    )


reset_keras()
model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=metrics,
)
model.summary()

In [ ]:
ckpt_path = os.path.join(CKPT_ROOT, "best_pneumonia_mnist_cnn.keras")
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=epochs,
    class_weight=class_weight,
    verbose=1,
    callbacks=make_callbacks(ckpt_path),
)
best_model = tf.keras.models.load_model(ckpt_path)
summary_graphics(history, best_model, test_ds)

## Tuning the decision threshold

In [ ]:
# Choose the decision threshold on the VALIDATION set (never on test), then apply it to
# test. A lower threshold trades a little precision for far fewer false negatives.
val_probs = best_model.predict(validation_ds).ravel()
precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
threshold = float(thresholds[np.argmax(f1)])
print(f"Tuned threshold: {threshold:.3f}  (default is 0.5)\n")

test_probs = best_model.predict(test_ds).ravel()
for name, thr in [("default 0.5", 0.5), (f"tuned {threshold:.3f}", threshold)]:
    pred = (test_probs >= thr).astype("int32")
    tp = int(((pred == 1) & (y_test == 1)).sum())
    fn = int(((pred == 0) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    print(
        f"{name:>14}: accuracy={np.mean(pred == y_test):.3f}  "
        f"recall={tp / (tp + fn):.3f}  precision={tp / (tp + fp):.3f}  "
        f"false negatives={fn}"
    )

show_confusion_matrix(
    y_test,
    (test_probs >= threshold).astype("int32"),
    title=f"Confusion Matrix (threshold = {threshold:.3f})",
)

## Grad-CAM: where is the model looking?

Grad-CAM weights the last convolutional feature map by the gradient of the pneumonia score,
showing which regions drove the prediction. On the most confident pneumonia case a healthy
model should favor the lung region over the image borders.

In [ ]:
# Grad-CAM on the model's most confident PNEUMONIA case. At 28x28 the localization is
# coarse, but a healthy model should still favor the lung region over the borders.
scores = best_model.predict(test_ds).ravel()
pos_i = max((i for i in range(len(y_test)) if y_test[i] == 1), key=lambda i: scores[i])
all_images = np.concatenate([bx.numpy() for bx, _ in test_ds], axis=0)

show_gradcam(
    all_images[pos_i],
    best_model,
    class_name="pneumonia",
    true_label="pneumonia",
    title="Grad-CAM: most confident pneumonia case",
)

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} "
    f"(duration: {notebook_end_time - notebook_start_time})"
)